# Winning Combinations

Each combo explicitly lists which mechanisms are turned on.

| combo | Δ from baseline | tasks | seeds | runs |
|---|---|---|---|---|
| revise+JEPA | +revise +jepa(0.1) | cifar10, mazes | 5 | 10 |
| revise+sparsity | +revise +topk(0.5) | sort, mazes | 5 | 10 |
| JEPA+sparsity | +jepa(0.1) +topk(0.5) | cifar10, sort | 5 | 10 |
| full stack | +revise +jepa(0.1) +topk(0.5) | cifar10, sort | 5 | 10 |

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

TASKS = ['sort', 'cifar10', 'mazes', 'parity']
SEEDS_MAIN = [0, 1, 2, 3, 4]
SEEDS_SWEEP = [0, 1, 2]

## Prior Results (single ideas)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    for name, stage, sweep in [('revise','st10','revise'), ('JEPA','st04','jepa_w0.1'),
                                 ('sparsity','st08','sparsity0.5')]:
        for task in ['cifar10','sort','mazes']:
            sub = df_prior[(df_prior.stage==stage)&(df_prior.sweep==sweep)&(df_prior.task==task)]
            if not sub.empty:
                m = sub.best_test_acc.mean()*100; bl = BASELINE_ACC[task]*100
                print(f'{name:10s} {task:10s}: {m:.1f}% ({m-bl:+.1f}pp)')
else:
    print('Prior data not found.')

## Combo Definitions

Each combo explicitly shows which parameters are added to the baseline.

In [ ]:
# Common building blocks
REVISE = dict(draft_mode='revise', draft_revise_weight=0.1,
              draft_corrupt_prob=0.15, draft_block_size=2)
JEPA = dict(cross_tick_jepa_weight=0.1, cross_tick_jepa_hidden_dim=128,
            cross_tick_jepa_predictor_depth=2, cross_tick_jepa_dropout=0.0)
SPARSITY = dict(topk_neurons=0.5)

COMBOS = [
    # (name, tasks, extra_params_dict)
    ('revise+jepa',     ['cifar10','mazes'], {**REVISE, **JEPA}),
    ('revise+sparsity', ['sort','mazes'],    {**REVISE, **SPARSITY}),
    ('jepa+sparsity',   ['cifar10','sort'],  {**JEPA, **SPARSITY}),
    ('full_stack',      ['cifar10','sort'],  {**REVISE, **JEPA, **SPARSITY}),
]

exps = []
for combo_name, tasks, extras in COMBOS:
    for task in tasks:
        module, base = BASE_CONFIGS[task]
        for s in [0,1,2,3,4]:
            tag = '+'.join(k[:3] for k in extras.keys() if k.startswith('draft'))[:10] or 'none'
            exps.append(Experiment(
                name=f'{task}_{combo_name}_s{s}',
                task=task, module=module,
                config={**base, 'seed': s, **extras}))
    print(f'{combo_name:20s}: {len(tasks)*5} runs on {tasks}')
print(f'\nTotal: {len(exps)} experiments')

# Show what each combo changes
for combo_name, _, extras in COMBOS:
    print(f'\n{combo_name}:')
    for k, v in extras.items():
        print(f'  + {k} = {v}')

## Run All

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/04_combos', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/04_combos')

In [ ]:
status('logs/deep/04_combos')

## Analysis

In [ ]:
df = collect('logs/deep/04_combos')
if df.empty:
    print('No results yet.')
else:
    df['combo'] = df['name'].apply(lambda n: '+'.join([p for p in ['revise','jepa','spar'] if p in n]))
    print(df[['name','task','combo','best_acc','delta']].to_string(index=False))
    plot_delta_bars(df, 'Combos vs baseline', 'figures/04_delta.png')
    print(significance_test(df).to_string(index=False))
    print(summary_stats(df, groupby=('combo','task')))